In [122]:
from pyspark.sql.functions import *

fact_orders = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/fact_orders"
)

dim_users = spark.read.parquet(
    "abfss://silver@stzomatoanalytics01.dfs.core.windows.net/dim_users"
)

restaurant_intelligence = spark.read.parquet(
    "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/restaurant_intelligence"
)

revenue_city = spark.read.parquet(
    "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/revenue_city"
)

In [123]:
total_revenue = (
    fact_orders
    .agg(sum("sales_amount"))
    .collect()[0][0]
)

In [124]:
total_orders = (
    fact_orders
    .agg(count("order_id"))
    .collect()[0][0]
)

In [125]:
total_customers = (
    fact_orders
    .select("user_id")
    .distinct()
    .count()
)

In [126]:
avg_order_value = (
    total_revenue / total_orders
)

In [127]:
active_customers = (
    fact_orders
    .groupBy("user_id")
    .count()
    .filter(col("count") >= 2)
    .count()
)

In [128]:
top_city = (
    revenue_city
    .orderBy(desc("revenue"))
    .first()["city"]
)

In [129]:
top_restaurant = (
    restaurant_intelligence
    .orderBy(desc("total_revenue"))
    .first()["name"]
)

In [130]:
executive_kpis = spark.createDataFrame(
    [(
        float(total_revenue),
        int(total_orders),
        int(total_customers),
        float(avg_order_value),
        int(active_customers),
        str(top_city),
        str(top_restaurant)
    )],
    [
        "total_revenue",
        "total_orders",
        "total_customers",
        "avg_order_value",
        "active_customers",
        "top_city",
        "top_restaurant"
    ]
)

In [131]:
display(executive_kpis)

In [132]:
executive_kpis.write \
    .mode("overwrite") \
    .parquet(
        "abfss://gold@stzomatoanalytics01.dfs.core.windows.net/executive_kpis"
    )